![Image](./snapshot_strategy.png)

1차에서 수행한 100개의 질문을 제외하고 다시 랜덤하게 100개 추출


In [ ]:
# pip install openpyxl 

In [1]:
import psycopg2
import pandas as pd
import numpy as np
import pickle
import lib.preprocess.preprocess as pp
import lib.preprocess.SectionExtractor as se
import re
import datetime
import pandas as pd
import re
import numpy as np
from sklearn import metrics
import os


In [2]:
htmlp = pp.HTMLParser()
codep = pp.CodeSectionParser()
ts = se.SectionExtractor()

In [3]:
file_list = os.listdir('.')

In [4]:
file_list = [x for x in file_list if x.endswith('snapshop2_sample.csv')]

In [5]:
df_q = pd.DataFrame()
for file in file_list :
    df_q = pd.concat([df_q, pd.read_csv(file, index_col=False)], axis = 0)

In [6]:
df_q[['id', 'question']]

,id,question
0,70264389,<Title>Population pyramid with seaborn python<...
1,70274885,"<Title>insert or update on table ""django_admin..."
2,70313318,<Title>perform upsert operation on postgres li...
3,70540832,<Title>Python/SQL - Connecting to different da...
4,70693775,<Title>How to pass an object to a process crea...
...,...,...
103,70731352,<Title>Python str not list</Title>. <Question>...
104,74047007,<Title>How to detect black contour in image us...
105,78840333,<Title>How to get every combination possible i...
106,71281263,<Title>Loop through a directory and add filena...


In [7]:
path_list = [f'../golden_dataset/{x}' for x in ['2nd', '3rd', '5th']]

In [8]:
a_list = []
for path in path_list : 
    a_list.append([f'{path}/{x}' for x in os.listdir(path) if x.endswith('.xlsx') and not x.startswith('~')])


In [9]:
a_list

[['../golden_dataset/2nd/22_jw_2nd_snapshop2_sample_annotation.xlsx',
  '../golden_dataset/2nd/22_jh_2nd_snapshop2_sample_annotation_jihyang.xlsx',
  '../golden_dataset/2nd/22_hj_2nd_snapshop2_sample_annotation_HJ.xlsx',
  '../golden_dataset/2nd/22_mk_2nd_snapshop2_sample_annotation.xlsx'],
 ['../golden_dataset/3rd/3rd_snapshop2_sample_annotation_Jihyang.xlsx',
  '../golden_dataset/3rd/3rd_snapshop2_sample_HJ.xlsx',
  '../golden_dataset/3rd/3rd_snapshop2_sample_annotation_mk.xlsx',
  '../golden_dataset/3rd/3rd_snapshop2_sample_jw.xlsx'],
 ['../golden_dataset/5th/5th_snapshop2_sample_annotation_jw.xlsx',
  '../golden_dataset/5th/5th_snapshop2_sample_annotation_Jihyang.xlsx',
  '../golden_dataset/5th/5th_snapshop2_sample_annotation_hj.xlsx',
  '../golden_dataset/5th/5th_snapshop2_sample_annotation_mk.xlsx']]

In [10]:
mapping = {'Basic': '<Difficulty Level>0</Difficulty Level>', 
           'Intermediate': '<Difficulty Level>1</Difficulty Level>', 
           'Advanced' : '<Difficulty Level>2</Difficulty Level>'}

In [11]:
tot_df = pd.DataFrame()
for a in a_list:
    df_0 = pd.read_excel(f'{a[0]}', engine='openpyxl')
    df_1 = pd.read_excel(f'{a[1]}', engine='openpyxl')
    df_2 = pd.read_excel(f'{a[2]}', engine='openpyxl')
    df_3 = pd.read_excel(f'{a[3]}', engine='openpyxl')

    df_0 = df_0[['id', 'answer']].rename(columns={'answer' : 'a_jh'})
    df_1 = df_1[['id', 'answer']].rename(columns={'answer' : 'a_hj'})
    df_2 = df_2[['id', 'answer']].rename(columns={'answer' : 'a_jw'})
    df_3 = df_3[['id', 'answer']].rename(columns={'answer' : 'a_mk'})

    df_0['a_jh'] = df_0['a_jh'].map(mapping)
    df_1['a_hj'] = df_1['a_hj'].map(mapping)
    df_2['a_jw'] = df_2['a_jw'].map(mapping)
    df_3['a_mk'] = df_3['a_mk'].map(mapping)

    df_m = df_0.merge(df_1, on='id') \
                .merge(df_2, on='id') \
                .merge(df_3, on='id')
    
    # df_m['sum'] = (df_m['a_jh']==df_m['a_hj'])&(df_m['a_hj'] ==df_m['a_jw']) & (df_m['a_jw']==df_m['a_mk'])
    # df_c = df_m[df_m['sum'] ==True]
    tot_df = pd.concat([tot_df, df_m], axis = 0)

In [12]:
tot_df = tot_df.melt(id_vars = 'id' , value_vars= ['a_jh', 'a_hj', 'a_jw', 'a_mk'], var_name='annotator', value_name='answer')

In [17]:
tot_df = tot_df.groupby(['id', 'answer']).count().reset_index().rename(columns = {'annotator' : 'count'})

In [18]:
tot_df[tot_df['id'] ==79245166]

,id,answer,count
551,79245166,<Difficulty Level>0</Difficulty Level>,1
552,79245166,<Difficulty Level>1</Difficulty Level>,3


In [19]:
tot_df[tot_df['count'] == 3]

# 1	70217014	<Difficulty Level>1</Difficulty Level>	3
# 4	70264812	<Difficulty Level>0</Difficulty Level>	3
# 10	70325197	<Difficulty Level>1</Difficulty Level>	3
# 18	70507747	<Difficulty Level>0</Difficulty Level>	3
# 20	70545853	<Difficulty Level>1</Difficulty Level>	3
# ...	...	...	...
# 532	79092651	<Difficulty Level>1</Difficulty Level>	3
# 538	79122866	<Difficulty Level>1</Difficulty Level>	3
# 542	79161167	<Difficulty Level>1</Difficulty Level>	3
# 547	79188432	<Difficulty Level>1</Difficulty Level>	3
# 552	79245166	<Difficulty Level>1</Difficulty Level>	3


,id,answer,count
1,70217014,<Difficulty Level>1</Difficulty Level>,3
4,70264812,<Difficulty Level>0</Difficulty Level>,3
10,70325197,<Difficulty Level>1</Difficulty Level>,3
18,70507747,<Difficulty Level>0</Difficulty Level>,3
20,70545853,<Difficulty Level>1</Difficulty Level>,3
...,...,...,...
532,79092651,<Difficulty Level>1</Difficulty Level>,3
538,79122866,<Difficulty Level>1</Difficulty Level>,3
542,79161167,<Difficulty Level>1</Difficulty Level>,3
547,79188432,<Difficulty Level>1</Difficulty Level>,3


In [20]:
golden_3_df = tot_df[tot_df['count'] ==3]

In [23]:
golden_3_df.shape

(144, 3)

In [21]:
len(golden_3_df['id'].unique())

144

In [24]:
golden_3_df['answer'].value_counts()

answer
<Difficulty Level>1</Difficulty Level>    99
<Difficulty Level>0</Difficulty Level>    26
<Difficulty Level>2</Difficulty Level>    19
Name: count, dtype: int64

In [25]:
df_golden = pd.merge(df_q[['id', 'question']], golden_3_df[['id', 'answer']], on = 'id')

In [26]:
df_golden

,id,question,answer
0,70325197,"<Title>Add a z3 constraint, such that the valu...",<Difficulty Level>1</Difficulty Level>
1,70556190,<Title>How to narrow testing examples in `torc...,<Difficulty Level>1</Difficulty Level>
2,70811411,<Title>How to loop through rows of specific co...,<Difficulty Level>1</Difficulty Level>
3,71102874,<Title>Python automatically converts the numer...,<Difficulty Level>1</Difficulty Level>
4,71594371,<Title>Pandas if-else condition</Title>. <Ques...,<Difficulty Level>0</Difficulty Level>
...,...,...,...
139,75050847,"<Title>I get results only for the last class, ...",<Difficulty Level>1</Difficulty Level>
140,74091080,<Title>Apply colored marker over an image</Tit...,<Difficulty Level>1</Difficulty Level>
141,76618653,<Title>How to handle multiple bound pairs for ...,<Difficulty Level>1</Difficulty Level>
142,77180527,<Title>ModuleNotFoundError: No module named 'k...,<Difficulty Level>1</Difficulty Level>


In [ ]:
file_path = "/mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty" 
df_golden.to_csv(f'{file_path}/q_output_code_y_snapshot2_validation.csv', index=False) 

In [ ]:
df_golden[df_golden['id'] == 73981914]

In [ ]:
file_path = f"{path_list['data_root_dir']}/result/annotate_difficulty"  

df_golden.to_csv(f'{file_path}/q_output_code_y_snapshot2_md.csv', index=False)
# /mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/q_output_code_y_snapshot2_md.csv

